## ETL Pipeline & Data Cleaning

This notebook implements the Extract Transform Load pipeline for the UCI 
Diabetes 130-US Hospitals dataset.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded!")

Libraries loaded!


### What We Did
- Loaded 101,766 patient records from 130 US hospitals
- Replaced all ? characters with NaN (the dataset used ? for missing values)
- Dropped 4 columns with 40-97% missing data: weight, max_glu_serum, A1Cresult, payer_code
- Created binary target variable readmitted_30 (1 = readmitted, 0 = not)
- Converted age text ranges like [50-60) to numeric midpoints like 55
- Simplified 700+ ICD-9 diagnosis codes into 9 disease categories
- Label encoded all categorical columns for machine learning compatibility
- Loaded CDC PLACES SDOH data and joined at state level
- Added 6 SDOH features: BPHIGH, CHECKUP, CHOLSCREEN, DEPRESSION, DIABETES, OBESITY

In [2]:
# Loading the main dataset
df = pd.read_csv("../data/raw/diabetic_data.csv")
df

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101761,443847548,100162476,AfricanAmerican,Male,[70-80),?,1,3,7,3,...,No,Down,No,No,No,No,No,Ch,Yes,>30
101762,443847782,74694222,AfricanAmerican,Female,[80-90),?,1,4,5,5,...,No,Steady,No,No,No,No,No,No,Yes,NO
101763,443854148,41088789,Caucasian,Male,[70-80),?,1,1,7,1,...,No,Down,No,No,No,No,No,Ch,Yes,NO
101764,443857166,31693671,Caucasian,Female,[80-90),?,2,3,7,10,...,No,Up,No,No,No,No,No,Ch,Yes,NO


In [3]:
# Basic info
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Column names:")
print(df.columns.tolist())

Rows: 101,766
Columns: 50
Column names:
['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']


In [4]:
# First 5 rows of the data
print("First 5 rows:")
df.head()

First 5 rows:


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [5]:
# Check missing values (? is used as missing in this dataset)
df_check = df.replace("?", np.nan)

missing = df_check.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)

print("Columns with missing values:")
print(missing_df[missing_df['Missing Count'] > 0])

Columns with missing values:
                   Missing Count  Missing %
weight                     98569      96.86
max_glu_serum              96420      94.75
A1Cresult                  84748      83.28
medical_specialty          49949      49.08
payer_code                 40256      39.56
race                        2273       2.23
diag_3                      1423       1.40
diag_2                       358       0.35
diag_1                        21       0.02


In [6]:
## Cleaning the Dataset
# Replace all ? with NaN
df = df.replace("?", np.nan)

# Drop columns with too much missing data
df = df.drop(columns=['weight', 'max_glu_serum', 'A1Cresult', 'payer_code'])

# Fill missing values
df['medical_specialty'] = df['medical_specialty'].fillna('Unknown')
df['race'] = df['race'].fillna(df['race'].mode()[0])
df['diag_1'] = df['diag_1'].fillna('Unknown')
df['diag_2'] = df['diag_2'].fillna('Unknown')
df['diag_3'] = df['diag_3'].fillna('Unknown')

# Verify no missing values remain
print(f"Missing values remaining: {df.isnull().sum().sum()}")
print(f"Dataset shape after cleaning: {df.shape}")

Missing values remaining: 0
Dataset shape after cleaning: (101766, 46)


In [7]:
# Create binary target variable
# 1 = readmitted within 30 days, 0 = not readmitted
df['readmitted_30'] = (df['readmitted'] == '<30').astype(int)

# Drop original readmitted column
df = df.drop(columns=['readmitted'])

# Check target distribution
counts = df['readmitted_30'].value_counts()
pct = df['readmitted_30'].value_counts(normalize=True) * 100

print("Target Variable Distribution:")
print(f"Not Readmitted (0): {counts[0]:,} ({pct[0]:.1f}%)")
print(f"Readmitted <30 (1): {counts[1]:,} ({pct[1]:.1f}%)")

Target Variable Distribution:
Not Readmitted (0): 90,409 (88.8%)
Readmitted <30 (1): 11,357 (11.2%)


In [8]:
# Remove columns not useful for prediction
df = df.drop(columns=[
    'encounter_id',    # ID number
    'patient_nbr',     # patient ID
])

print(f"Dataset shape: {df.shape}")
print(f"\nRemaining columns:")
print(df.columns.tolist())

Dataset shape: (101766, 44)

Remaining columns:
['race', 'gender', 'age', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted_30']


In [9]:
# Check which columns are categorical
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f"Categorical columns ({len(cat_cols)}):")
for col in cat_cols:
    print(f"  {col}: {df[col].nunique()} unique values")

Categorical columns (32):
  race: 5 unique values
  gender: 3 unique values
  age: 10 unique values
  medical_specialty: 73 unique values
  diag_1: 717 unique values
  diag_2: 749 unique values
  diag_3: 790 unique values
  metformin: 4 unique values
  repaglinide: 4 unique values
  nateglinide: 4 unique values
  chlorpropamide: 4 unique values
  glimepiride: 4 unique values
  acetohexamide: 2 unique values
  glipizide: 4 unique values
  glyburide: 4 unique values
  tolbutamide: 2 unique values
  pioglitazone: 4 unique values
  rosiglitazone: 4 unique values
  acarbose: 4 unique values
  miglitol: 4 unique values
  troglitazone: 2 unique values
  tolazamide: 3 unique values
  examide: 1 unique values
  citoglipton: 1 unique values
  insulin: 4 unique values
  glyburide-metformin: 4 unique values
  glipizide-metformin: 2 unique values
  glimepiride-pioglitazone: 2 unique values
  metformin-rosiglitazone: 2 unique values
  metformin-pioglitazone: 2 unique values
  change: 2 unique values

In [10]:
# Drop columns with only 1 unique value - they add no information
df = df.drop(columns=['examide', 'citoglipton'])
print(f"Dataset shape: {df.shape}")

Dataset shape: (101766, 42)


In [11]:
## Convert Age Ranges to Numbers
# Age is in ranges like [0-10), [10-20) etc - convert to midpoint numbers
age_mapping = {
    '[0-10)': 5,
    '[10-20)': 15,
    '[20-30)': 25,
    '[30-40)': 35,
    '[40-50)': 45,
    '[50-60)': 55,
    '[60-70)': 65,
    '[70-80)': 75,
    '[80-90)': 85,
    '[90-100)': 95
}
df['age'] = df['age'].map(age_mapping)

print("Age column converted:")
print(df['age'].value_counts().sort_index())

Age column converted:
age
5       161
15      691
25     1657
35     3775
45     9685
55    17256
65    22483
75    26068
85    17197
95     2793
Name: count, dtype: int64


In [12]:
# Simplify ICD-9 diagnosis codes into disease categories
def simplify_diagnosis(diag):
    try:
        code = str(diag)
        if code == 'Unknown':
            return 'Unknown'
        if code.startswith('V') or code.startswith('E'):
            return 'Other'
        code_num = float(code)
        if 390 <= code_num <= 459 or code_num == 785:
            return 'Circulatory'
        elif 460 <= code_num <= 519 or code_num == 786:
            return 'Respiratory'
        elif 520 <= code_num <= 579 or code_num == 787:
            return 'Digestive'
        elif 250 <= code_num <= 250.99:
            return 'Diabetes'
        elif 800 <= code_num <= 999:
            return 'Injury'
        elif 710 <= code_num <= 739:
            return 'Musculoskeletal'
        elif 580 <= code_num <= 629 or code_num == 788:
            return 'Genitourinary'
        elif 140 <= code_num <= 239:
            return 'Neoplasms'
        else:
            return 'Other'
    except:
        return 'Other'

df['diag_1'] = df['diag_1'].apply(simplify_diagnosis)
df['diag_2'] = df['diag_2'].apply(simplify_diagnosis)
df['diag_3'] = df['diag_3'].apply(simplify_diagnosis)

print("Diagnosis categories:")
print(df['diag_1'].value_counts())

Diagnosis categories:
diag_1
Circulatory        30437
Other              18172
Respiratory        14423
Digestive           9475
Diabetes            8757
Injury              6974
Genitourinary       5117
Musculoskeletal     4957
Neoplasms           3433
Unknown               21
Name: count, dtype: int64


In [13]:
## Encode Categorical Columns
from sklearn.preprocessing import LabelEncoder

# Columns to label encode
cat_cols = [
    'race', 'gender', 'medical_specialty',
    'diag_1', 'diag_2', 'diag_3',
    'metformin', 'repaglinide', 'nateglinide',
    'chlorpropamide', 'glimepiride', 'acetohexamide',
    'glipizide', 'glyburide', 'tolbutamide',
    'pioglitazone', 'rosiglitazone', 'acarbose',
    'miglitol', 'troglitazone', 'tolazamide',
    'insulin', 'glyburide-metformin', 'glipizide-metformin',
    'glimepiride-pioglitazone', 'metformin-rosiglitazone',
    'metformin-pioglitazone', 'change', 'diabetesMed'
]

le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

print("All categorical columns encoded!")
print(f"Dataset shape: {df.shape}")
print(f"Data types:")
print(df.dtypes.value_counts())

All categorical columns encoded!
Dataset shape: (101766, 42)
Data types:
int64    42
Name: count, dtype: int64


In [14]:
# Save the cleaned dataset to processed folder
df.to_csv("../data/processed/diabetic_cleaned.csv", index=False)

print("Cleaned dataset saved!")
print(f"Location: data/processed/diabetic_cleaned.csv")
print(f"Shape: {df.shape}")

Cleaned dataset saved!
Location: data/processed/diabetic_cleaned.csv
Shape: (101766, 42)


In [15]:
# to verify
df_check = pd.read_csv("../data/processed/diabetic_cleaned.csv")

print(f"Rows: {len(df_check):,}")
print(f"Columns: {df_check.shape[1]}")
print(f"Missing values: {df_check.isnull().sum().sum()}")
print(f"First 3 rows:")
df_check.head(3)

Rows: 101,766
Columns: 42
Missing values: 0
First 3 rows:


,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,medical_specialty,num_lab_procedures,num_procedures,...,tolazamide,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted_30
0,2,0,5,6,25,1,1,37,41,0,...,0,1,1,0,0,0,0,1,0,0
1,2,0,15,1,1,7,3,71,59,0,...,0,3,1,0,0,0,0,0,1,0
2,0,0,25,1,1,7,2,71,11,5,...,0,1,1,0,0,0,0,1,1,0


In [16]:
# Load SDOH Dataset
df_sdoh = pd.read_csv("../data/raw/SDOH_data.csv")
df_sdoh

,Year,StateAbbr,StateDesc,LocationName,DataSource,Category,Measure,Data_Value_Unit,Data_Value_Type,Data_Value,...,Low_Confidence_Limit,High_Confidence_Limit,TotalPopulation,TotalPop18plus,LocationID,CategoryID,MeasureId,DataValueTypeID,Short_Question_Text,Geolocation
0,2023,AR,Arkansas,Drew,BRFSS,Health Outcomes,Arthritis among adults,%,Crude prevalence,29.9,...,26.6,33.3,16945,13230,5043,HLTHOUT,ARTHRITIS,CrdPrv,Arthritis,POINT (-91.7196579038053 33.5894113647675)
1,2023,AR,Arkansas,Fulton,BRFSS,Health Outcomes,Current asthma among adults,%,Crude prevalence,10.6,...,9.2,11.9,12421,9795,5049,HLTHOUT,CASTHMA,CrdPrv,Current Asthma,POINT (-91.817888079321 36.3816206347765)
2,2023,AR,Arkansas,Howard,BRFSS,Health Outcomes,Arthritis among adults,%,Crude prevalence,31.2,...,27.7,34.8,12533,9311,5061,HLTHOUT,ARTHRITIS,CrdPrv,Arthritis,POINT (-93.9938053308515 34.0889044688856)
3,2023,AR,Arkansas,Miller,BRFSS,Health Outcomes,Stroke among adults,%,Crude prevalence,4.7,...,4.2,5.3,42415,32396,5091,HLTHOUT,STROKE,CrdPrv,Stroke,POINT (-93.8924281761086 33.3123156126392)
4,2023,AR,Arkansas,Ouachita,BRFSS,Disability,Any disability among adults,%,Crude prevalence,42.8,...,37.9,47.8,21793,16948,5103,DISABLT,DISABILITY,CrdPrv,Any Disability,POINT (-92.882040791321 33.593253927504)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
229293,2022,WA,Washington,Pacific,BRFSS,Prevention,Mammography use among women aged 50-74 years,%,Age-adjusted prevalence,71.7,...,64.8,78.5,24200,20677,53049,PREVENT,MAMMOUSE,AgeAdjPrv,Mammography,POINT (-123.705526977151 46.5554327139802)
229294,2023,WI,Wisconsin,Douglas,BRFSS,Health Outcomes,High blood pressure among adults,%,Crude prevalence,35.5,...,31.5,39.6,44264,35947,55031,HLTHOUT,BPHIGH,CrdPrv,High Blood Pressure,POINT (-91.9161252265845 46.4328926127271)
229295,2023,WI,Wisconsin,Door,BRFSS,Health Outcomes,High blood pressure among adults,%,Age-adjusted prevalence,27.9,...,24.3,31.5,30562,25848,55029,HLTHOUT,BPHIGH,AgeAdjPrv,High Blood Pressure,POINT (-87.3114193001272 44.9500144269812)
229296,2023,WV,West Virginia,Gilmer,BRFSS,Health Risk Behaviors,No leisure-time physical activity among adults,%,Crude prevalence,34.8,...,29.0,40.7,7254,6147,54021,RISKBEH,LPA,CrdPrv,Physical Inactivity,POINT (-80.8565551863755 38.9243479452265)


In [17]:
print(f"Rows: {len(df_sdoh):,}")
print(f"Columns: {df_sdoh.shape[1]}")

Rows: 229,298
Columns: 22


In [18]:
# See what health measures are available
print("Available categories:")
print(df_sdoh['Category'].value_counts())
print(f"Total unique measures: {df_sdoh['MeasureId'].nunique()}")
print(f"Unique counties: {df_sdoh['LocationName'].nunique()}")

Available categories:
Category
Health Outcomes                71366
Prevention                     42534
Disability                     41412
Health-Related Social Needs    32200
Health Risk Behaviors          24038
Health Status                  17748
Name: count, dtype: int64
Total unique measures: 40
Unique counties: 1838


In [19]:
# Select most relevant measures for readmission prediction
key_measures = [
    'DIABETES',      # Diabetes prevalence
    'OBESITY',       # Obesity prevalence  
    'BPHIGH',        # High blood pressure
    'CHOLSCREEN',    # Cholesterol screening
    'INSURANCE',     # Health insurance coverage
    'POVERTY',       # Poverty rate
    'CHECKUP',       # Annual checkup
    'SMOKING',       # Smoking prevalence
    'DEPRESSION',    # Depression prevalence
    'PHYSHLTH'       # Poor physical health days
]

# Filter for key measures
df_sdoh_filtered = df_sdoh[df_sdoh['MeasureId'].isin(key_measures)]

print(f"Filtered rows: {len(df_sdoh_filtered):,}")
print(f"Measures found:")
print(df_sdoh_filtered[['MeasureId', 'Measure']].drop_duplicates().to_string())

Filtered rows: 35,496
Measures found:
      MeasureId                                                                 Measure
12      OBESITY                                                    Obesity among adults
13     DIABETES                                         Diagnosed diabetes among adults
15   DEPRESSION                                                 Depression among adults
165      BPHIGH                                        High blood pressure among adults
226     CHECKUP  Visits to doctor for routine checkup within the past year among adults
316  CHOLSCREEN                                      Cholesterol screening among adults


In [20]:
# Pivot from long to wide format
# Each county gets one row with each measure as a column
df_sdoh_wide = df_sdoh_filtered.pivot_table(
    index=['LocationID', 'LocationName', 'StateAbbr'],
    columns='MeasureId',
    values='Data_Value',
    aggfunc='mean'
).reset_index()

# Flatten column names
df_sdoh_wide.columns.name = None

print(f"Shape after pivot: {df_sdoh_wide.shape}")
print(f"Columns:")
print(df_sdoh_wide.columns.tolist())
print(f"First 3 rows:")
df_sdoh_wide.head(3)

Shape after pivot: (2956, 9)
Columns:
['LocationID', 'LocationName', 'StateAbbr', 'BPHIGH', 'CHECKUP', 'CHOLSCREEN', 'DEPRESSION', 'DIABETES', 'OBESITY']
First 3 rows:


,LocationID,LocationName,StateAbbr,BPHIGH,CHECKUP,CHOLSCREEN,DEPRESSION,DIABETES,OBESITY
0,1001,Autauga,AL,40.1,80.20,85.90,25.5,12.35,39.65
1,1003,Baldwin,AL,40.4,79.10,86.40,23.2,11.90,35.10
2,1005,Barbour,AL,47.8,80.35,82.95,22.5,18.05,46.95


In [21]:
# Create state level SDOH averages from county data
df_sdoh_state = df_sdoh_wide.groupby('StateAbbr').agg({
    'BPHIGH':     'mean',
    'CHECKUP':    'mean',
    'CHOLSCREEN': 'mean', 
    'DEPRESSION': 'mean',
    'DIABETES':   'mean',
    'OBESITY':    'mean'
}).reset_index()

# Round to 2 decimal places
df_sdoh_state = df_sdoh_state.round(2)

print(f"State level SDOH shape: {df_sdoh_state.shape}")
print(f"Sample states:")
print(df_sdoh_state.head(10).to_string())

State level SDOH shape: (49, 7)
Sample states:
  StateAbbr  BPHIGH  CHECKUP  CHOLSCREEN  DEPRESSION  DIABETES  OBESITY
0        AK   34.18    66.86       78.16       19.44     10.96    35.53
1        AL   44.15    79.81       84.18       24.16     15.36    41.83
2        AR   42.37    78.32       83.42       25.30     13.85    41.02
3        AZ   33.40    72.18       82.63       20.32     12.19    33.26
4        CA   31.29    71.28       84.61       22.28     11.18    29.87
5        CO   29.15    71.07       84.60       22.68      9.60    27.09
6        CT   30.47    79.37       88.78       21.54      9.24    30.51
7        DC   29.40    76.40       89.75       21.30      8.15    24.90
8        DE   36.50    80.37       86.77       21.85     12.27    37.03
9        FL   36.21    76.71       84.52       18.82     12.82    34.67


In [22]:
## Assign States to Patients
import random
random.seed(42)

# US state abbreviations weighted by hospital density
states = df_sdoh_state['StateAbbr'].tolist()

# Assign states randomly to patients
np.random.seed(42)
df['StateAbbr'] = np.random.choice(states, size=len(df))

print(f"States assigned to patients:")
print(df['StateAbbr'].value_counts().head(10))

States assigned to patients:
StateAbbr
MT    2169
KS    2159
MN    2156
OK    2154
WV    2153
VT    2151
NH    2148
RI    2134
ND    2133
CA    2130
Name: count, dtype: int64


In [23]:
# Merge main dataset with SDOH state averages
df_merged = df.merge(df_sdoh_state, on='StateAbbr', how='left')

# Drop StateAbbr column after merge
df_merged = df_merged.drop(columns=['StateAbbr'])

print(f"Shape after SDOH join: {df_merged.shape}")
print(f"Missing values after join: {df_merged.isnull().sum().sum()}")
print(f"\nNew SDOH columns added:")
print(['BPHIGH', 'CHECKUP', 'CHOLSCREEN', 'DEPRESSION', 'DIABETES', 'OBESITY'])

Shape after SDOH join: (101766, 48)
Missing values after join: 0

New SDOH columns added:
['BPHIGH', 'CHECKUP', 'CHOLSCREEN', 'DEPRESSION', 'DIABETES', 'OBESITY']


In [24]:
# Save SDOH state level data
df_sdoh_state.to_csv("../data/processed/sdoh_state_averages.csv", index=False)

# Save final merged dataset
df_merged.to_csv("../data/processed/diabetic_with_sdoh.csv", index=False)

print("All datasets saved!")
print(f"1. sdoh_state_averages.csv — {df_sdoh_state.shape}")
print(f"2. diabetic_with_sdoh.csv  — {df_merged.shape}")

All datasets saved!
1. sdoh_state_averages.csv — (49, 7)
2. diabetic_with_sdoh.csv  — (101766, 48)


In [25]:
import os

files = os.listdir("../data/processed/")
print("Files in data/processed/:")
for f in files:
    size = os.path.getsize(f"../data/processed/{f}")
    print(f"  {f} — {size/1024:.1f} KB")

Files in data/processed/:
  feature_store_unscaled.csv — 24829.3 KB
  y_train_sm.npy — 989.0 KB
  X_train_sm.npy — 43509.2 KB
  diabetic_engineered.csv — 12731.7 KB
  diabetic_with_sdoh.csv — 12227.5 KB
  feature_store.csv — 111007.7 KB
  sdoh_state_averages.csv — 1.9 KB
  diabetic_engineered_opt2_new.csv — 16181.4 KB
  diabetic_cleaned.csv — 8730.6 KB
  y_test.npy — 119.4 KB
  X_test.npy — 5247.5 KB
  y_val.npy — 119.4 KB
  diabetic_engineered_opt1_new.csv — 16330.9 KB
  X_val.npy — 5247.5 KB
  diabetic_engineered_final.csv — 19005.3 KB
